# Notebook 12: The Model Context Protocol (MCP)

## Connecting AI Applications to the World, the Standard Way

So far in this series, every tool our agents used was written *inside* the same Python
process as the agent. That works for tutorials — but real companies have hundreds of data
sources and tools, and hundreds of AI applications that want to use them. **MCP** is the
open standard that fixes this.

In this notebook you will learn, from zero:

1. **Why MCP exists** — the integration problem it solves
2. **How MCP works** — hosts, clients, servers, and the protocol's primitives
3. **The MCP Inspector** — the official tool for testing and debugging MCP servers
4. **Building your own MCP server** with FastMCP (tools, resources, prompts)
5. **Using MCP tools from a LangGraph agent** — the payoff

**Prerequisites**: Notebooks 01-11, `OPENAI_API_KEY` in `.env`, and Node.js 22+ (`npx`) for
the Inspector sections
**Time**: 45-55 minutes

---
# Part 0: Why Does MCP Exist?

## The N×M Integration Problem

Imagine a company with **M AI applications** (a chat assistant, a coding assistant, a support
bot...) and **N external systems** (Slack, GitHub, a database, Google Drive...). Before MCP,
connecting them all meant writing **M × N custom integrations** — every app needed bespoke
code for every system, and none of it was reusable.

**MCP (Model Context Protocol)** turns this into **M + N**: a system implements an MCP
*server* once, and every MCP-compatible *client* can use it immediately.

> **The USB-C analogy** (from the official docs): just as USB-C gives electronic devices one
> standard connector, MCP gives AI applications one standard way to connect to external
> systems.

## Who Benefits?

| Audience | Benefit |
|---|---|
| **Developers** | Build a server once; every MCP host (Claude, ChatGPT, VS Code, Cursor...) can use it |
| **AI applications** | Instant access to an ecosystem of ready-made servers (files, databases, SaaS APIs) |
| **End users** | More capable assistants that can actually *do things* with their data |

MCP was open-sourced by Anthropic in November 2024 and is now supported across the industry.

In [1]:
import os
import shutil
import subprocess
import sys
import warnings
from dotenv import load_dotenv

load_dotenv()
warnings.filterwarnings("ignore")  # keep outputs clean (e.g. LangChain beta notice)

# ── Library versions ─────────────────────────────────────────────────────────
from importlib.metadata import version
print("=== Environment ===")
for pkg in ["fastmcp", "mcp", "langchain", "langgraph"]:
    print(f"  {pkg:<10} {version(pkg)}")

# ── Node.js check (required only for the Inspector sections) ────────────────
NODE_OK = shutil.which("npx") is not None
if NODE_OK:
    node_ver = subprocess.run(["node", "--version"], capture_output=True, text=True).stdout.strip()
    print(f"\nNode.js: {node_ver} ✅ (MCP Inspector sections will run)")
else:
    print("\nNode.js: ❌ not found — Inspector cells will be skipped (install Node 22+ to run them)")

# ── Model for the LangGraph agent in Part 3 ──────────────────────────────────
MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
print(f"LLM: {MODEL}")

# Working directory for the servers we will write
os.makedirs("servers", exist_ok=True)

=== Environment ===
  fastmcp    4.2.5
  mcp        2.3.0
  langchain  1.5.2
  langgraph  1.2.24

Node.js: v22.22.3 ✅ (MCP Inspector sections will run)
LLM: gpt-4o-mini


---
# Part 1: How MCP Works

## The Architecture: Host → Clients → Servers

MCP follows a **client-server architecture**:

- **MCP Host** — the AI application (Claude Desktop, VS Code, *your LangGraph agent*). It
  coordinates everything.
- **MCP Client** — a component the host creates *per server*. Each client keeps one dedicated
  connection to one server.
- **MCP Server** — a program that exposes capabilities (tools, data, prompts) over the
  protocol. It can run locally (as a subprocess) or remotely (over HTTP).

```
┌──────────────── MCP Host (your AI app) ────────────────┐
│  MCP Client 1        MCP Client 2        MCP Client 3  │
└───────┬──────────────────┬────────────────────┬────────┘
        │ dedicated        │ dedicated          │ shared remote endpoint
        ▼                  ▼                    ▼
   MCP Server A       MCP Server B        MCP Server C (remote)
   (local, stdio)     (local, stdio)      (Streamable HTTP)
```

## Two Layers

| Layer | What it defines |
|---|---|
| **Data layer** (inner) | The JSON-RPC 2.0 protocol: discovery, primitives (tools/resources/prompts), notifications |
| **Transport layer** (outer) | How bytes move: **stdio** (local subprocess, 1 client) or **Streamable HTTP** (remote, many clients, optional SSE streaming) |

The transport is interchangeable — the same JSON-RPC messages flow over both.

## The Data Layer: Stateless JSON-RPC 2.0

Every MCP request is a self-contained JSON-RPC 2.0 message. MCP is **stateless**: each
request carries the protocol version, client identity, and capabilities in a `_meta` field,
so the server never has to remember previous requests.

A client first *discovers* what a server offers:

```json
// Request — every request carries _meta
{"jsonrpc": "2.0", "id": 1, "method": "server/discover",
 "params": {"_meta": {"io.modelcontextprotocol/protocolVersion": "2026-07-28",
                      "io.modelcontextprotocol/clientInfo": {"name": "example-client", "version": "1.0.0"}}}}

// Response — the server's identity, versions, and capabilities
{"jsonrpc": "2.0", "id": 1,
 "result": {"supportedVersions": ["2026-07-28"],
            "capabilities": {"tools": {"listChanged": true}, "resources": {}},
            "_meta": {"io.modelcontextprotocol/serverInfo": {"name": "example-server", "version": "1.0.0"}}}}
```

You will rarely write these messages by hand — SDKs handle them — but recognizing the shape
helps when you debug with the Inspector later.

## The Primitives

**Server primitives** (what servers expose):

| Primitive | Purpose | Methods |
|---|---|---|
| **Tools** | Executable functions the AI can invoke (API calls, DB queries, actions) | `tools/list`, `tools/call` |
| **Resources** | Data sources the AI can read (files, records, configs) | `resources/list`, `resources/read` |
| **Prompts** | Reusable prompt templates with arguments | `prompts/list`, `prompts/get` |

**Client primitive**: **elicitation** — a server can pause a tool call and ask the *user* for
more input (e.g. "are you sure you want to delete this?").

The core pattern is always **discover, then use**: `*/list` first, then `*/call`, `*/read`,
or `*/get`.

## Hands-On: Your First MCP Server & Client (In-Process)

Let's make this concrete. `FastMCP` is the standard Python framework for building MCP
servers *and* clients. We define a tiny server right here — one tool, one resource, one
prompt — and talk to it with a `fastmcp.Client` **in-process** (no subprocess, no network),
so you can see the protocol round-trips clearly.

> The official `mcp` SDK also bundles a FastMCP (`mcp.server.fastmcp.FastMCP`); the
> standalone `fastmcp` package we use here is the actively developed framework that
> LangChain's MCP integration is built on.

In [2]:
from fastmcp import FastMCP, Client

# ── Define an MCP server: decorators turn Python functions into MCP primitives ──
demo = FastMCP("Demo Server")


@demo.tool
def add(a: int, b: int) -> int:
    """Add two numbers together."""
    return a + b


@demo.resource("config://app")
def app_config() -> str:
    """The application's current configuration."""
    return "environment=development, debug=true"


@demo.prompt
def code_review(code: str) -> str:
    """Generate a code review request for the given code."""
    return f"Please review this code for bugs and style issues:\n\n{code}"


print("Demo server defined: 1 tool (add), 1 resource (config://app), 1 prompt (code_review)")

Demo server defined: 1 tool (add), 1 resource (config://app), 1 prompt (code_review)


In [3]:
# ── Connect with an in-process client and use every primitive ────────────────
async with Client(demo) as client:
    # 1. DISCOVER tools — the client asks "what can you do?"
    tools = await client.list_tools()
    print("=== tools/list ===")
    for t in tools:
        params = list(t.input_schema.get("properties", {}).keys())
        print(f"  {t.name}({', '.join(params)})  — {t.description}")

    # 2. USE the tool
    result = await client.call_tool("add", {"a": 19, "b": 23})
    print(f"\n=== tools/call add(19, 23) ===")
    print(f"  result: {result.content[0].text}")

    # 3. Read the resource
    resource = await client.read_resource("config://app")
    print(f"\n=== resources/read config://app ===")
    print(f"  {resource[0].text}")

    # 4. Get the prompt
    prompt = await client.get_prompt("code_review", {"code": "x = 1"})
    print(f"\n=== prompts/get code_review ===")
    print(f"  {prompt.messages[0].content.text}")

print("\n✅ You just performed the full MCP flow: discover → list → call/read/get")

=== tools/list ===
  add(a, b)  — Add two numbers together.



=== tools/call add(19, 23) ===
  result: 42

=== resources/read config://app ===
  environment=development, debug=true

=== prompts/get code_review ===
  Please review this code for bugs and style issues:

x = 1

✅ You just performed the full MCP flow: discover → list → call/read/get


---
# Part 2: The MCP Inspector

The [MCP Inspector](https://modelcontextprotocol.io/docs/2026-07-28/tools/inspector.md) is
the official developer tool for testing and debugging MCP servers. It ships as one npm
package (`@modelcontextprotocol/inspector`) with **three clients** behind one binary:

| Client | Launch | Best for |
|---|---|---|
| **Web** | `npx @modelcontextprotocol/inspector <server command>` | Full graphical UI in the browser (the default, richest surface) |
| **CLI** | `... --cli <server command>` | Scripts, CI pipelines, and — as we'll see — running *from a notebook* |
| **TUI** | `... --tui <server command>` | Interactive terminal UI when no browser is available |

All three share the same core: same transports, same config files, same protocol-era
negotiation. There's also `mcpdo`, an experimental connection client that connects once and
runs many commands against the open connection.

> **Requirements**: Node.js 22+ (`npx`). The web client opens a **tokenized URL** and stores
> OAuth secrets in the OS keychain — treat Inspector sessions as privileged.

## Our Test Subject: A Custom Ticket Server

To exercise the Inspector (and later, LangGraph), we use a realistic MCP server that ships
with this repo: [`servers/ticket_server.py`](servers/ticket_server.py) — a support-ticket
system with:

- **Tools**: `get_ticket`, `list_open_tickets`, `create_ticket`
- **Resource**: `tickets://open` — a live summary of open tickets
- **Prompt**: `triage_ticket` — a template for triaging a ticket

The whole server is ~60 lines. The key pattern:

```python
from fastmcp import FastMCP

mcp = FastMCP("Ticket Server")

@mcp.tool
def get_ticket(ticket_id: int) -> dict:
    """Look up a support ticket by its ID."""   # ← docstring = tool description for the LLM
    ...

@mcp.resource("tickets://open")
def open_tickets_summary() -> str: ...

@mcp.prompt
def triage_ticket(ticket_text: str) -> str: ...

if __name__ == "__main__":
    mcp.run(transport="stdio")
```

**The docstring becomes the tool description and the type hints become the JSON input
schema** — that is exactly what an LLM reads to call your tool correctly. Open the file and
read it before continuing.

In [4]:
# Sanity-check the server file: import it in-process and list what it exposes
sys_path_added = "servers" not in sys.path
if sys_path_added:
    sys.path.insert(0, "servers")
from ticket_server import mcp as _check_mcp

async with Client(_check_mcp) as c:
    names = [t.name for t in await c.list_tools()]
    resources = [str(r.uri) for r in await c.list_resources()]
    prompts = [p.name for p in await c.list_prompts()]

print("✅ servers/ticket_server.py loads and exposes:")
print(f"   tools:     {names}")
print(f"   resources: {resources}")
print(f"   prompts:   {prompts}")

✅ servers/ticket_server.py loads and exposes:
   tools:     ['get_ticket', 'list_open_tickets', 'create_ticket']
   resources: ['tickets://open']
   prompts:   ['triage_ticket']


## Inspector CLI: Scriptable Testing

The CLI mode is perfect for automation — and for running the Inspector *from inside this
notebook*. The pattern is:

```bash
npx @modelcontextprotocol/inspector --cli <how-to-launch-the-server> --method <mcp-method> [args]
```

For a local **stdio** server, "how to launch" is just the command that starts it. Let's list
the tools our ticket server exposes:

In [5]:
PY = sys.executable  # the .venv python, so the server has fastmcp available

if NODE_OK:
    !npx -y @modelcontextprotocol/inspector --cli {PY} servers/ticket_server.py --method tools/list 2>/dev/null
else:
    print("Skipped — Node.js not available")

{
  "tools": [
    {
      "name": "get_ticket",
      "title": "Get Ticket",
      "description": "Look up a support ticket by its ID.",
      "inputSchema": {
        "type": "object",
        "properties": {
          "ticket_id": {
            "type": "integer"
          }
        },
        "required": [
          "ticket_id"
        ],
        "additionalProperties": false
      },
      "outputSchema": {
        "type": "object",
        "additionalProperties": true
      },
      "_meta": {
        "fastmcp": {
          "tags": []
        }
      }
    },
    {
      "name": "list_open_tickets",
      "title": "List Open Tickets",
      "description": "List all tickets whose status is 'open'.",
      "inputSchema": {
        "type": "object",
        "properties": {},
        "additionalProperties": false
      },
      "outputSchema": {
        "type": "object",
        "properties": {
          "result": {
            "items": {
              "additionalProperties": true,
  

## Calling Tools, Reading Resources, Getting Prompts

Same pattern for every method — only `--method` and its arguments change:

- `tools/call` → `--tool-name <name> --tool-arg key=value`
- `resources/read` → `--uri <resource-uri>`
- `prompts/get` → `--prompt-name <name> --prompt-args key=value`

Add `--format json | jq ...` to pipe results into other tools (shown below).

In [6]:
if NODE_OK:
    print("── tools/call get_ticket ──────────────────────────────")
    !npx -y @modelcontextprotocol/inspector --cli {PY} servers/ticket_server.py \
        --method tools/call --tool-name get_ticket --tool-arg ticket_id=1024 2>/dev/null

    print("\n── resources/read tickets://open ──────────────────────")
    !npx -y @modelcontextprotocol/inspector --cli {PY} servers/ticket_server.py \
        --method resources/read --uri "tickets://open" 2>/dev/null

    print("\n── prompts/get triage_ticket ──────────────────────────")
    !npx -y @modelcontextprotocol/inspector --cli {PY} servers/ticket_server.py \
        --method prompts/get --prompt-name triage_ticket --prompt-args ticket_text="My invoice is wrong" 2>/dev/null
else:
    print("Skipped — Node.js not available")

── tools/call get_ticket ──────────────────────────────


{
  "content": [
    {
      "type": "text",
      "text": "{\"ticket_id\":1024,\"subject\":\"Charged twice this month\",\"category\":\"billing\",\"status\":\"open\",\"priority\":\"high\"}"
    }
  ],
  "structuredContent": {
    "ticket_id": 1024,
    "subject": "Charged twice this month",
    "category": "billing",
    "status": "open",
    "priority": "high"
  },
  "isError": false
}



── resources/read tickets://open ──────────────────────


{
  "contents": [
    {
      "uri": "tickets://open",
      "mimeType": "text/plain",
      "text": "Open tickets:\n#1024 Charged twice this month (high)\n#1025 App crashes on iOS 17 (high)"
    }
  ]
}



── prompts/get triage_ticket ──────────────────────────


{
  "description": "Create a triage prompt for a raw customer message.",
  "messages": [
    {
      "role": "user",
      "content": {
        "type": "text",
        "text": "Triage this customer message. Classify it as billing, technical, feature, or general, assign a priority (low/medium/high), and suggest the first response.\n\nMessage: My invoice is wrong"
      }
    }
  ]
}


In [7]:
# ── Machine-readable output for CI: JSON + jq ────────────────────────────────
if NODE_OK:
    print("Tool names via --format json | jq:")
    !npx -y @modelcontextprotocol/inspector --cli {PY} servers/ticket_server.py \
        --method tools/list --format json 2>/dev/null | jq -r '.result.tools[].name'
    print("\nExit code of a successful call: ", end="")
    !npx -y @modelcontextprotocol/inspector --cli {PY} servers/ticket_server.py \
        --method tools/list > /dev/null 2>&1; echo $?
else:
    print("Skipped — Node.js not available")

Tool names via --format json | jq:


get_ticket
list_open_tickets
create_ticket



Exit code of a successful call: 

0


## Inspector Web & TUI: The Graphical Experience

For interactive exploration, launch the **web client** — it prints a tokenized URL and opens
your browser:

```bash
# stdio server
npx @modelcontextprotocol/inspector python servers/ticket_server.py

# remote HTTP server (once we start one in Part 3)
npx @modelcontextprotocol/inspector --server-url http://127.0.0.1:8123/mcp --transport http
```

**A tour of the web UI:**

1. **Connect panel** — pick the transport, launch command or URL, then *Connect*
2. **Tools tab** — list tools, fill in arguments in a form, run them, see results
3. **Resources tab** — browse and read resources by URI
4. **Prompts tab** — render prompt templates with arguments
5. **History sidebar** — every JSON-RPC request/response, exactly as exchanged (invaluable
   for debugging)
6. **Protocol era indicator** — legacy vs modern (2026-07-28) operation

The **TUI** (`--tui`) offers the same tabs in the terminal with keyboard navigation — handy
over SSH.

> **Try it**: run the web command above in a terminal, connect, and call `get_ticket` with
> `ticket_id=1025`. Watch the JSON-RPC exchange in the history sidebar — that is exactly the
> protocol you learned in Part 1.

---
# Part 3: Serving Over HTTP & Using MCP Tools in LangGraph

So far our server ran over **stdio** — perfect for local, single-client use (Claude Desktop,
the Inspector). For a server many clients share, MCP uses **Streamable HTTP**: the server
listens on a URL (conventionally ending in `/mcp`) and any number of clients connect.

FastMCP speaks both transports with the *same server object* — only `mcp.run(...)` changes:

```python
mcp.run(transport="stdio")                          # local subprocess
mcp.run(transport="http", host=..., port=...)       # remote, streamable HTTP
```

Let's serve our ticket server over HTTP (in a background thread, so the notebook keeps
running) and connect to it over the network.

In [8]:
import threading
import time

from ticket_server import mcp as ticket_mcp  # the FastMCP instance from our file

MCP_PORT = 8123


def _serve():
    ticket_mcp.run(transport="http", host="127.0.0.1", port=MCP_PORT, show_banner=False)


server_thread = threading.Thread(target=_serve, daemon=True)
server_thread.start()

# Wait until the server answers
from fastmcp import Client as FClient

for attempt in range(30):
    try:
        async with FClient(f"http://127.0.0.1:{MCP_PORT}/mcp") as c:
            await c.ping()
        break
    except Exception:
        time.sleep(0.5)

print(f"✅ Ticket server listening on http://127.0.0.1:{MCP_PORT}/mcp (Streamable HTTP)")

[11/24/26 16:52:18] INFO     Starting MCP server 'Ticket Server' with transport 'http' on          ]8;id=8494551;file:///Users/sourangshupal/Downloads/langraph-tutorials/.venv/lib/python3.12/site-packages/fastmcp/server/mixins/transport.py\transport.py]8;;\:]8;id=8494552;file:///Users/sourangshupal/Downloads/langraph-tutorials/.venv/lib/python3.12/site-packages/fastmcp/server/mixins/transport.py#358\358]8;;\
                             http://127.0.0.1:8123/mcp                                                             

INFO:     Started server process [68856]


INFO:     Waiting for application startup.


INFO:     Application startup complete.


INFO:     Uvicorn running on http://127.0.0.1:8123 (Press CTRL+C to quit)


INFO:     127.0.0.1:58801 - "POST /mcp HTTP/1.1" 200 OK


... (56 outputs left out)

INFO:     127.0.0.1:59788 - "POST /mcp HTTP/1.1" 404 Not Found


INFO:     127.0.0.1:59832 - "POST /mcp HTTP/1.1" 200 OK


INFO:     127.0.0.1:59832 - "POST /mcp HTTP/1.1" 404 Not Found


✅ Ticket server listening on http://127.0.0.1:8123/mcp (Streamable HTTP)


In [9]:
# ── A remote client, over real HTTP ──────────────────────────────────────────
async with FClient(f"http://127.0.0.1:{MCP_PORT}/mcp") as client:
    tools = await client.list_tools()
    print("Tools over HTTP:", [t.name for t in tools])

    result = await client.call_tool("list_open_tickets", {})
    print("\nOpen tickets, served remotely:")
    print(" ", result.content[0].text)

INFO:     127.0.0.1:59850 - "POST /mcp HTTP/1.1" 200 OK


INFO:     127.0.0.1:59850 - "POST /mcp HTTP/1.1" 200 OK


Tools over HTTP: ['get_ticket', 'list_open_tickets', 'create_ticket']
INFO:     127.0.0.1:59850 - "POST /mcp HTTP/1.1" 200 OK



Open tickets, served remotely:
  [{"ticket_id":1024,"subject":"Charged twice this month","category":"billing","status":"open","priority":"high"},{"ticket_id":1025,"subject":"App crashes on iOS 17","category":"technical","status":"open","priority":"high"}]


## The Payoff: A LangGraph Agent with MCP Tools

This is why MCP matters for this course. MCP support now lives **inside LangChain** in the
`langchain.mcp` namespace (built on FastMCP — the standalone
`langchain-mcp-adapters` package is deprecated):

```python
from langchain.mcp import MCPAdapter

async with MCPAdapter({"mcpServers": {"tickets": {"url": "http://.../mcp"}}}) as adapter:
    tools = await adapter.list_tools()   # MCP tools, converted to LangChain tools
```

Hand those tools to `create_agent` and the LLM can discover *and call* your MCP server on its
own. One `mcpServers` dict can hold **many** servers (stdio or HTTP) — tools are namespaced
automatically.

In [10]:
from langchain.agents import create_agent

with warnings.catch_warnings():
    warnings.filterwarnings("ignore", message=".*langchain.mcp.*")
    from langchain.mcp import MCPAdapter

MCP_CONFIG = {
    "mcpServers": {
        "tickets": {"url": f"http://127.0.0.1:{MCP_PORT}/mcp"},
        # Add more servers here — stdio entries use "command"/"args" instead of "url":
        # "filesystem": {"command": "npx", "args": ["-y", "@modelcontextprotocol/server-filesystem", "/tmp"]},
    }
}

async with MCPAdapter(MCP_CONFIG) as adapter:
    mcp_tools = await adapter.list_tools()
    print("LangChain tools from MCP:", [t.name for t in mcp_tools])

    agent = create_agent(
        f"openai:{MODEL}",
        mcp_tools,
        system_prompt="You are a support agent. Use the ticket tools to answer questions accurately.",
    )
    response = await agent.ainvoke({
        "messages": [("user", "What open tickets do we have, and what is ticket 1024 about?")]
    })

print("\n=== Agent conversation ===")
for msg in response["messages"]:
    role = msg.__class__.__name__.replace("Message", "")
    tool_calls = getattr(msg, "tool_calls", None)
    if tool_calls:
        calls = ", ".join(f"{tc['name']}({tc['args']})" for tc in tool_calls)
        print(f"  [{role}] → calls {calls}")
    elif msg.content:
        text = msg.content if isinstance(msg.content, str) else str(msg.content)
        print(f"  [{role}] {text[:200]}")

INFO:     127.0.0.1:59924 - "POST /mcp HTTP/1.1" 200 OK


INFO:     127.0.0.1:59924 - "POST /mcp HTTP/1.1" 200 OK


LangChain tools from MCP: ['get_ticket', 'list_open_tickets', 'create_ticket']


INFO:     127.0.0.1:59924 - "POST /mcp HTTP/1.1" 200 OK


INFO:     127.0.0.1:60048 - "POST /mcp HTTP/1.1" 200 OK



=== Agent conversation ===
  [Human] What open tickets do we have, and what is ticket 1024 about?
  [AI] → calls list_open_tickets({}), get_ticket({'ticket_id': 1024})
  [Tool] [{'type': 'text', 'text': '[{"ticket_id":1024,"subject":"Charged twice this month","category":"billing","status":"open","priority":"high"},{"ticket_id":1025,"subject":"App crashes on iOS 17","category
  [Tool] [{'type': 'text', 'text': '{"ticket_id":1024,"subject":"Charged twice this month","category":"billing","status":"open","priority":"high"}', 'id': 'lc_283e1a5a-5108-4d99-b64e-e2c71453b965'}]
  [AI] We currently have the following open tickets:

1. **Ticket ID:** 1024
   - **Subject:** Charged twice this month
   - **Category:** Billing
   - **Status:** Open
   - **Priority:** High

2. **Ticket I


## Production Notes

- **Tool errors**: when an MCP tool fails (e.g. "ticket not found"), the error reaches the
  model as a `ToolMessage` with `status="error"` — the agent sees it and can recover,
  instead of your app crashing.
- **Elicitation**: if a server asks the user for input mid-tool-call, `langchain.mcp` surfaces
  it as a LangGraph `interrupt` — resume with `Command(resume={...})` (Notebook 05 patterns).
- **Transports**: stdio for local servers shipped to users' machines; Streamable HTTP for
  shared/remote servers. The old HTTP+SSE and WebSocket transports are deprecated.
- **Security**: MCP servers run with your privileges — only connect to servers you trust,
  keep the Inspector's tokenized URL private, and validate tool inputs on the server side.

---
## Summary

| Concept | What it is | Where you saw it |
|---|---|---|
| Host / Client / Server | The three MCP participants | Part 1 diagram |
| Data layer | Stateless JSON-RPC 2.0 with `_meta` on every request | Annotated messages |
| Tools / Resources / Prompts | Server primitives: act, read, template | Demo server + ticket server |
| Elicitation | Server asks the user for input mid-call | Production notes |
| stdio / Streamable HTTP | The two transports | Parts 2 & 3 |
| MCP Inspector | Web / CLI / TUI testing tool | `npx ... --cli` runs |
| FastMCP | Python framework for servers & clients | `@mcp.tool` etc. |
| `langchain.mcp.MCPAdapter` | MCP tools → LangGraph agent | Part 3 agent |

**Links**

- [MCP Documentation](https://modelcontextprotocol.io/docs/2026-07-28/getting-started/intro.md)
- [MCP Inspector](https://modelcontextprotocol.io/docs/2026-07-28/tools/inspector.md)
- [FastMCP](https://gofastmcp.com/docs/getting-started/quickstart.md)
- [LangChain MCP integration](https://docs.langchain.com/oss/python/migrate/langchain-mcp-adapters)
- [Official MCP servers](https://github.com/modelcontextprotocol/servers)

**Next**: Notebook 13 — the A2A protocol: when your *agents* (not just tools) need to talk
to each other.

In [11]:
# Background HTTP server is a daemon thread — it stops when the kernel exits.
print("✅ Notebook complete. Ticket server thread will end with the kernel.")

✅ Notebook complete. Ticket server thread will end with the kernel.
